<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/Sankey_01/sankey1_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sankey diagram 4
### Policy Source → Target Group → Target Class → Target (Global)

#Import libaries

In [1]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors

In [2]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [3]:
# Helper function to fetch data from Airtable
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [4]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel2 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel3 = fetch_airtable_data("tblGwx3sWdVfvlMto")  # Target Class 

In [5]:
# Fjern rækker med manglende værdier

# For tabel0: behold kun rækker hvor både 'Policy Source' og 'Targets (policy targets)' er udfyldt
tabel0 = tabel0.dropna(subset=['Policy source','Targets (policy targets)'])

# For tabel1: behold kun rækker hvor både 'Target name', 'Target Group' og 'Target Class' er udfyldt
tabel1 = tabel1.dropna(subset=['Target name','Target Group','Target Class'])

# For tabel2: behold kun rækker hvor både 'Target Group' og 'Targets' er udfyldt
tabel2 = tabel2.dropna(subset=['Target Group', 'Targets'])

# For tabel3: behold kun rækker hvor både 'Name' og 'Targets' er udfyldt
tabel3 = tabel3.dropna(subset=['Name','Targets'])


In [6]:
# Explode and rename
t0 = tabel0.copy().explode("Targets (policy targets)").rename(columns={"Targets (policy targets)": "target_id"})

t1a = tabel1.copy().explode("Target Group").rename(columns={"Target Group": "target_group_id"})

t1b = tabel1.copy().explode("Target Class").rename(columns={"Target Class": "target_class_id"})

t2 = tabel2.copy().explode("Targets").rename(columns={"Targets": "target_id_from_group"})

t3 = tabel3.copy().explode("Targets").rename(columns={"Targets": "target_id_from_class"})



In [7]:
# Filtrér direkte før registrering
allowed_policies = [
    "Technical Summary: Land Use and Climate Change",
    "Convention on Biological Diversity",
    "Common approach to integrating biodiversity and nature-based solutions for sustainable development into the United Nations policy and programme planning and delivery",
    #"European Green Deal",
    #"Common Agricultural Policy - Strategic plan 2023-2027",
    #"EU biodiversity strategy 2030",
    #"Aftale om et grønt Danmark (grøn trepart)",
    #"Mere, bedre og større natur i Danmark",
    #"Vandområdeplaner 2021-2027"
 ]
t0 = t0[t0["Policy source"].isin(allowed_policies)]

In [8]:
# Register in DuckDB
duckdb.register("tabel0", t0)       # Policy source → target_id
duckdb.register("tabel1a", t1a)     # target_id → target_group_id
duckdb.register("tabel1b", t1b)     # target_id → target_class_id
duckdb.register("tabel2", t2)       # target_group_id → target_id_from_group
duckdb.register("tabel3", t3)       # target_class_id → target_id_from_class


In [9]:
query = """
SELECT
    t0."Policy source"              AS policy_source,
    t1_start."Target name"          AS target_name_1,
    t2."Target Group"               AS target_group,
    t1_mid."Target name"            AS target_name_2,
    t3."Name"                       AS target_class,
    t1_end."Target name"            AS target_name_3

FROM
    tabel0 t0

-- Join første target (fra Policy Source)
JOIN tabel1 t1_start
    ON t0.target_id = t1_start.id

-- Join Target Group-link
JOIN tabel1a t1a
    ON t0.target_id = t1a.id

JOIN tabel2 t2
    ON t1a.target_group_id = t2.id

-- Find navn på det target, som group peger på
JOIN tabel1 t1_mid
    ON t2.target_id_from_group = t1_mid.id

-- Join Target Class-link
JOIN tabel1b t1b
    ON t0.target_id = t1b.id

JOIN tabel3 t3
    ON t1b.target_class_id = t3.id

-- Find navn på det target, som class peger på
JOIN tabel1 t1_end
    ON t3.target_id_from_class = t1_end.id
"""

results = duckdb.sql(query).df()


In [10]:
#  print("Antal rækker i resultatet:", len(results))
#  results.head(10) 

In [11]:
# Forkort og saml labels
results['policy_source'] = results['policy_source'].apply(lambda x: x[:70] + '…' if isinstance(x, str) and len(x) > 70 else x)
results['target_name_3'] = results['target_name_3'].apply(lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 40 else x)

results["final_target"] = results["target_name_3"]

#Udarbejd labels
source_labels = results['policy_source']
middle_labels = results['target_group']
class_labels = results['target_class']
target_labels = results['target_name_3'] 

all_labels = pd.concat([source_labels, middle_labels, class_labels, target_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}

In [12]:
# print(results[["target_name_3", "final_target"]].head(5))


In [13]:
# Forbindelser
# Link 1: policy_source → target_group
links1 = pd.DataFrame({
    'source': source_labels.map(label_to_index).values,
    'target': middle_labels.map(label_to_index).values,
    'value': 1
})

# Link 2: target_group → target_class
links2 = pd.DataFrame({
    'source': middle_labels.map(label_to_index).values,
    'target': class_labels.map(label_to_index).values,
    'value': 1
})

# Link 3: target_class → target_name_3
links3 = pd.DataFrame({
    'source': class_labels.map(label_to_index).values,
    'target': target_labels.map(label_to_index).values,
    'value': 1
})

# combine all links
all_links = pd.concat([links1, links2, links3], ignore_index=True)


In [14]:
# Brug Plotlys palette
node_colors = plotly.colors.qualitative.Plotly

# Tildel farver til unikke labels (noder)
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}

# Liste med farver i samme rækkefølge som unique_labels
node_colors_list = [color_map[label] for label in unique_labels]


# Funktion til at lysne farver (uden brug af gennemsigtighed)
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lysnet farve til hver link ud fra source-node
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in all_links['source']]



# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color=link_colors
    )
)])
fig.update_layout(title_text="Policy Source → Target Group → Target Class → Target (Global)", font_size=12, height=3000, width=4000)
fig.show()

In [15]:
# Gem som interaktiv HTML-fil
# fig.write_html("sankey_diagram_filtered.html")

# Download i Colab
# from google.colab import files
# files.download("sankey_diagram_filtered.html")